# Kimi Researcher — Experiment Analysis

Analyze all Kimi experiment results across CPU and GPU platforms, 3 trials per seed.

In [ ]:
import json
import glob
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from collections import defaultdict
from pathlib import Path

matplotlib.rcParams['figure.figsize'] = (12, 6)
matplotlib.rcParams['font.size'] = 11

BASE = Path('..') if Path('../results').exists() else Path('.')
print(f'Base: {BASE.resolve()}')

## 1. Data Collection

In [ ]:
import json as _jsonfrom collections import defaultdictdef normalize_seed_name(raw):    """Strip _gpu, _cpu, timestamps, then title-case."""    s = raw    parts = s.rsplit('_', 2)    if len(parts) >= 3 and parts[-2].isdigit() and parts[-1].isdigit():        s = '_'.join(parts[:-2])    return (s.replace('_gpu', '').replace('_cpu', '')             .replace('_', ' ').title()             .replace('Ai For', 'AI for')             .replace('Of Learned Representations', 'of Learned Repr.')             .replace('And Benchmarks', '& Benchmarks')             .replace('And Cleaning', '& Cleaning')             .replace('In Machine Learning', 'in ML')             .replace('Representation Learning', 'Repr. Learning'))def get_trial_key(path_str):    """Return (normalized_seed, trial_id) for dedup."""    after = path_str.split('/kimi/')[1].split('/')    if '/results/' in path_str:        return (normalize_seed_name(after[0]), after[1])    elif after[0].startswith('run_'):        return (normalize_seed_name(after[1]), after[0].replace('run_', 'trial'))    else:        return (normalize_seed_name(after[0]), 'trial1')rows = []seen_trials = set()# results/ directory (organized trials)for seed_dir in sorted((BASE / 'results' / 'kimi').iterdir()):    if not seed_dir.is_dir(): continue    seed_raw = seed_dir.name    platform = 'gpu' if '_gpu' in seed_raw else 'cpu'    seed = normalize_seed_name(seed_raw)        for trial_dir in sorted(seed_dir.iterdir()):        if not trial_dir.is_dir() or not trial_dir.name.startswith('trial'): continue        trial_num = int(trial_dir.name.replace('trial',''))                dedup_key = (seed, trial_dir.name)        if dedup_key in seen_trials: continue        seen_trials.add(dedup_key)                review_files = sorted(trial_dir.glob('code/idea_*/reviews.json'))        if not review_files: continue        try:            reviews = _json.loads(review_files[-1].read_text())        except: continue                row = {'seed': seed, 'platform': platform, 'trial': trial_num,               'avg_score': reviews.get('avg_score', 0)}                for rev in reviews.get('reviews', []):            name = rev.get('source', '?')            short = name.replace('agent:', '')            row[f'reviewer_{short}'] = rev.get('overall_score')            for dim, val in rev.get('scores', {}).items():                if isinstance(val, (int, float)):                    row[f'dim_{dim}_{short}'] = val                # Tracker        tracker_file = trial_dir / 'tracker' / 'tracker.json'        if tracker_file.exists():            try:                tracker = _json.loads(tracker_file.read_text())                actions = tracker.get('actions', [])                for a in actions:                    stage = a.get('stage', '')                    if 'self_review' in stage:                        key = f'sr_{stage}_count'                        row[key] = row.get(key, 0) + 1                        if a.get('outcome') == 'success':                            row[f'sr_{stage}_pass'] = row.get(f'sr_{stage}_pass', 0) + 1                    elapsed = a.get('elapsed_seconds', 0)                    if elapsed > 0:                        key = f'time_{stage}'                        row[key] = row.get(key, 0) + elapsed            except: pass                summary_file = trial_dir / 'tracker' / 'summary.json'        if summary_file.exists():            try:                s = _json.loads(summary_file.read_text())                row['wall_time_h'] = s.get('wall_time_seconds', 0) / 3600                row['total_steps'] = s.get('total_steps', 0)                row['ideas_tried'] = s.get('ideas_tried', 0)            except: pass                rows.append(row)# outputs/kimi/run_N/ (GPU results not yet in results/)for run_dir in sorted((BASE / 'outputs' / 'kimi').glob('run_*')):    run_num = int(run_dir.name.split('_')[1])    for seed_dir in sorted(run_dir.iterdir()):        if not seed_dir.is_dir(): continue        seed = normalize_seed_name(seed_dir.name)                dedup_key = (seed, f'trial{run_num}')        if dedup_key in seen_trials: continue        seen_trials.add(dedup_key)                review_files = sorted(seed_dir.glob('idea_*/reviews.json'))        if not review_files: continue        try:            reviews = _json.loads(review_files[-1].read_text())        except: continue                row = {'seed': seed, 'platform': 'gpu', 'trial': run_num,               'avg_score': reviews.get('avg_score', 0)}        for rev in reviews.get('reviews', []):            name = rev.get('source', '?').replace('agent:', '')            row[f'reviewer_{name}'] = rev.get('overall_score')            for dim, val in rev.get('scores', {}).items():                if isinstance(val, (int, float)):                    row[f'dim_{dim}_{name}'] = val                tracker_file = seed_dir / 'tracker.json'        if tracker_file.exists():            try:                tracker = _json.loads(tracker_file.read_text())                for a in tracker.get('actions', []):                    stage = a.get('stage', '')                    if 'self_review' in stage:                        key = f'sr_{stage}_count'                        row[key] = row.get(key, 0) + 1                        if a.get('outcome') == 'success':                            row[f'sr_{stage}_pass'] = row.get(f'sr_{stage}_pass', 0) + 1                    elapsed = a.get('elapsed_seconds', 0)                    if elapsed > 0:                        row[f'time_{stage}'] = row.get(f'time_{stage}', 0) + elapsed            except: pass                summary_file = seed_dir / 'summary.json'        if summary_file.exists():            try:                s = _json.loads(summary_file.read_text())                row['wall_time_h'] = s.get('wall_time_seconds', 0) / 3600                row['total_steps'] = s.get('total_steps', 0)                row['ideas_tried'] = s.get('ideas_tried', 0)            except: pass        rows.append(row)# Timestamp-based outputs (trial 1 if not already seen)for ws in sorted((BASE / 'outputs' / 'kimi').iterdir()):    if not ws.is_dir() or ws.name.startswith('run_') or ws.name == 'logs': continue    parts = ws.name.rsplit('_', 2)    if len(parts) < 3 or not parts[-2].isdigit(): continue    seed = normalize_seed_name(ws.name)        dedup_key = (seed, 'trial1')    if dedup_key in seen_trials: continue    seen_trials.add(dedup_key)        review_files = sorted(ws.glob('idea_*/reviews.json'))    if not review_files: continue    try:        reviews = _json.loads(review_files[-1].read_text())    except: continue        row = {'seed': seed, 'platform': 'gpu', 'trial': 1,           'avg_score': reviews.get('avg_score', 0)}    for rev in reviews.get('reviews', []):        name = rev.get('source', '?').replace('agent:', '')        row[f'reviewer_{name}'] = rev.get('overall_score')        for dim, val in rev.get('scores', {}).items():            if isinstance(val, (int, float)):                row[f'dim_{dim}_{name}'] = val    rows.append(row)df = pd.DataFrame(rows)print(f'Total trials: {len(df)}')print(f'CPU: {len(df[df.platform=="cpu"])}, GPU: {len(df[df.platform=="gpu"])}')print(f'Seeds: {df.seed.nunique()}')print()for seed in sorted(df.seed.unique()):    n = len(df[df.seed == seed])    print(f'  {seed}: {n} trials')print()df.head()

## 2. Peer Review Scores by Seed

In [ ]:
# Clean seed names for displaydef clean_seed(s):    return (s.replace('_gpu', '').replace('_cpu', '')             .replace('_', ' ').title()             .replace('Ai For', 'AI for')             .replace('Nlp', 'NLP')             .replace('Of Learned Representations', '\nof Learned Repr.')             .replace('And Benchmarks', '\n& Benchmarks')             .replace('And Cleaning', '\n& Cleaning')             .replace('In Machine Learning', '\nin ML')             .replace('Representation Learning', '\nRepr. Learning'))fig, axes = plt.subplots(1, 2, figsize=(16, 7))for ax, platform in zip(axes, ['cpu', 'gpu']):    pdata = df[df.platform == platform].copy()    if len(pdata) == 0:        ax.set_title(f'Kimi — {platform.upper()} (no data)')        continue        seed_means = pdata.groupby('seed')['avg_score'].agg(['mean', 'sem']).sort_values('mean')    seeds = seed_means.index.tolist()    means = seed_means['mean'].values    sems = seed_means['sem'].fillna(0).values        colors = ['#4CAF50' if m >= 4 else '#FF9800' if m >= 3 else '#F44336' for m in means]    y = range(len(seeds))        bars = ax.barh(y, means, xerr=sems, color=colors, alpha=0.8, capsize=4, height=0.6)    ax.set_yticks(y)    ax.set_yticklabels([clean_seed(s) for s in seeds], fontsize=9)    ax.set_xlabel('Peer Review Score (avg ± SE)')    ax.set_title(f'Kimi — {platform.upper()} Seeds ({len(seeds)} seeds)', fontsize=12, fontweight='bold')    ax.axvline(x=8, color='green', linestyle='--', alpha=0.4, label='Accept (8)')    ax.axvline(x=4, color='gray', linestyle='--', alpha=0.3, label='Reject (<5)')    ax.set_xlim(0, 9)    ax.legend(fontsize=8, loc='lower right')        # Annotate with values    for j, (m, se) in enumerate(zip(means, sems)):        n = len(pdata[pdata.seed == seeds[j]])        ax.text(m + se + 0.15, j, f'{m:.1f} (n={n})', va='center', fontsize=8)plt.suptitle('Kimi Researcher — Peer Review Scores by Seed', fontsize=14, fontweight='bold', y=1.01)plt.tight_layout()plt.savefig('kimi_scores_by_seed.png', dpi=150, bbox_inches='tight')plt.show()# Print tablesummary = df.groupby(['platform', 'seed'])['avg_score'].agg(['mean', 'std', 'count']).round(2)summary['se'] = (summary['std'] / np.sqrt(summary['count'])).round(2)print(summary.to_string())print(f"\nOverall CPU: {df[df.platform=='cpu']['avg_score'].mean():.2f} ± {df[df.platform=='cpu']['avg_score'].sem():.2f}")print(f"Overall GPU: {df[df.platform=='gpu']['avg_score'].mean():.2f} ± {df[df.platform=='gpu']['avg_score'].sem():.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

for ax, platform in zip(axes, ['cpu', 'gpu']):
    pdata = df[df.platform == platform]
    seeds = sorted(pdata.seed.unique())
    means = [pdata[pdata.seed == s]['avg_score'].mean() for s in seeds]
    sems = [pdata[pdata.seed == s]['avg_score'].sem() for s in seeds]
    
    colors = ['#2196F3' if m >= 4 else '#F44336' for m in means]
    bars = ax.barh(range(len(seeds)), means, xerr=sems, color=colors, alpha=0.8, capsize=4)
    ax.set_yticks(range(len(seeds)))
    ax.set_yticklabels([s.replace(' ', '\n') if len(s) > 20 else s for s in seeds], fontsize=9)
    ax.set_xlabel('Peer Review Score')
    ax.set_title(f'Kimi — {platform.upper()} Seeds')
    ax.axvline(x=8, color='green', linestyle='--', alpha=0.5, label='Accept (8)')
    ax.axvline(x=5, color='orange', linestyle='--', alpha=0.5, label='Revision (5)')
    ax.set_xlim(0, 10)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('kimi_scores_by_seed.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Reviewer Bias Analysis

In [ ]:
reviewer_cols = [c for c in df.columns if c.startswith('reviewer_')]
reviewer_data = {}
for col in reviewer_cols:
    name = col.replace('reviewer_', '')
    scores = df[col].dropna()
    reviewer_data[name] = scores

fig, ax = plt.subplots(figsize=(10, 5))
positions = range(len(reviewer_data))
names = list(reviewer_data.keys())
bp = ax.boxplot([reviewer_data[n].values for n in names], positions=positions,
                patch_artist=True, widths=0.6)
colors = ['#2196F3', '#4CAF50', '#FF9800']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_xticks(positions)
ax.set_xticklabels(names)
ax.set_ylabel('Score')
ax.set_title('Reviewer Score Distributions (Kimi papers)')
ax.axhline(y=8, color='green', linestyle='--', alpha=0.3, label='Accept')
ax.axhline(y=4, color='red', linestyle='--', alpha=0.3, label='Reject')

for i, name in enumerate(names):
    scores = reviewer_data[name]
    ax.annotate(f'μ={scores.mean():.1f}', xy=(i, scores.mean()),
                xytext=(i+0.3, scores.mean()+0.3), fontsize=9)

ax.legend()
plt.tight_layout()
plt.savefig('kimi_reviewer_bias.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Per-Dimension Analysis

In [ ]:
dimensions = ['novelty', 'soundness', 'significance', 'clarity', 'reproducibility',
              'experimental_rigor', 'references', 'reference_integrity', 'results_integrity']

dim_means = {}
for dim in dimensions:
    cols = [c for c in df.columns if c.startswith(f'dim_{dim}_')]
    if cols:
        all_vals = pd.concat([df[c] for c in cols]).dropna()
        dim_means[dim] = all_vals.mean()

fig, ax = plt.subplots(figsize=(12, 5))
dims = list(dim_means.keys())
vals = [dim_means[d] for d in dims]
colors = ['#F44336' if v < 4 else '#FF9800' if v < 6 else '#4CAF50' for v in vals]
bars = ax.bar(range(len(dims)), vals, color=colors, alpha=0.8)
ax.set_xticks(range(len(dims)))
ax.set_xticklabels([d.replace('_', '\n') for d in dims], rotation=0, fontsize=9)
ax.set_ylabel('Mean Score (1-10)')
ax.set_title('Kimi Papers — Per-Dimension Scores (all reviewers)')
ax.axhline(y=6, color='green', linestyle='--', alpha=0.3)
ax.axhline(y=4, color='red', linestyle='--', alpha=0.3)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.1f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('kimi_dimension_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. CPU vs GPU Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))cpu_scores = df[df.platform == 'cpu']['avg_score']gpu_scores = df[df.platform == 'gpu']['avg_score']bp = ax.boxplot([cpu_scores.values, gpu_scores.values],                labels=['CPU', 'GPU'], patch_artist=True, widths=0.5)bp['boxes'][0].set_facecolor('#2196F3')bp['boxes'][1].set_facecolor('#FF9800')for box in bp['boxes']:    box.set_alpha(0.7)ax.set_ylabel('Peer Review Score')ax.set_title('Kimi: CPU vs GPU Performance')ax.axhline(y=8, color='green', linestyle='--', alpha=0.3, label='Accept')ax.legend()ax.annotate(f'μ={cpu_scores.mean():.2f}±{cpu_scores.sem():.2f}',            xy=(1, cpu_scores.mean()), xytext=(1.3, cpu_scores.mean()), fontsize=10)ax.annotate(f'μ={gpu_scores.mean():.2f}±{gpu_scores.sem():.2f}',            xy=(2, gpu_scores.mean()), xytext=(2.3, gpu_scores.mean()), fontsize=10)plt.tight_layout()plt.savefig('kimi_cpu_vs_gpu.png', dpi=150, bbox_inches='tight')plt.show()print(f'CPU: {cpu_scores.mean():.2f} ± {cpu_scores.sem():.2f} (n={len(cpu_scores)})')print(f'GPU: {gpu_scores.mean():.2f} ± {gpu_scores.sem():.2f} (n={len(gpu_scores)})')print(f'Diff: {cpu_scores.mean() - gpu_scores.mean():+.2f}')

## 6. Self-Review Impact

In [ ]:
sr_cols = [c for c in df.columns if c.startswith('sr_') and c.endswith('_count')]
sr_data = []
for col in sr_cols:
    gate = col.replace('sr_', '').replace('_count', '')
    counts = df[col].dropna()
    pass_col = col.replace('_count', '_pass')
    passes = df[pass_col].dropna() if pass_col in df.columns else pd.Series(dtype=float)
    sr_data.append({
        'gate': gate,
        'total_invocations': int(counts.sum()),
        'total_passes': int(passes.sum()) if len(passes) > 0 else 0,
        'mean_per_run': counts.mean(),
    })

sr_df = pd.DataFrame(sr_data)
sr_df['pass_rate'] = (sr_df['total_passes'] / sr_df['total_invocations'] * 100).round(1)
sr_df['revisions_per_run'] = (sr_df['mean_per_run'] - sr_df['total_passes'] / len(df)).round(1)
print(sr_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
gates = sr_df['gate'].tolist()
x = range(len(gates))
ax.bar(x, sr_df['mean_per_run'], color='#2196F3', alpha=0.7, label='Avg invocations/run')
ax.set_xticks(x)
ax.set_xticklabels([g.replace('self_review_', '') for g in gates])
ax.set_ylabel('Count')
ax.set_title('Self-Review: Average Invocations Per Run')
for i, row in sr_df.iterrows():
    ax.text(i, row['mean_per_run'] + 0.1, f"{row['pass_rate']}% pass", ha='center', fontsize=9)
ax.legend()
plt.tight_layout()
plt.savefig('kimi_self_review_impact.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Time Analysis

In [ ]:
time_cols = [c for c in df.columns if c.startswith('time_')]
time_data = []
for col in time_cols:
    stage = col.replace('time_', '')
    vals = df[col].dropna() / 60  # convert to minutes
    if len(vals) > 0:
        time_data.append({
            'stage': stage,
            'n': len(vals),
            'mean_min': vals.mean(),
            'se_min': vals.sem(),
            'min_min': vals.min(),
            'max_min': vals.max(),
        })

time_df = pd.DataFrame(time_data).sort_values('mean_min', ascending=False)
print(time_df.round(1).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
stages = time_df['stage'].tolist()
means = time_df['mean_min'].tolist()
sems = time_df['se_min'].tolist()
colors = ['#F44336' if m > 60 else '#FF9800' if m > 10 else '#4CAF50' for m in means]
ax.barh(range(len(stages)), means, xerr=sems, color=colors, alpha=0.8, capsize=3)
ax.set_yticks(range(len(stages)))
ax.set_yticklabels(stages)
ax.set_xlabel('Time (minutes)')
ax.set_title('Kimi — Time Spent Per Stage (total across all invocations)')
plt.tight_layout()
plt.savefig('kimi_time_per_stage.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Wall Time Distribution

In [ ]:
wt = df[df['wall_time_h'] > 0]

fig, ax = plt.subplots(figsize=(10, 5))
for platform, color in [('cpu', '#2196F3'), ('gpu', '#FF9800')]:
    data = wt[wt.platform == platform]['wall_time_h']
    if len(data) > 0:
        ax.hist(data, bins=15, alpha=0.6, color=color, label=f'{platform.upper()} (μ={data.mean():.1f}h)')

ax.set_xlabel('Wall Time (hours)')
ax.set_ylabel('Count')
ax.set_title('Kimi — Wall Time Distribution Per Run')
ax.legend()
plt.tight_layout()
plt.savefig('kimi_wall_time.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Score vs Time Correlation

In [ ]:
wt_scores = df[(df['wall_time_h'] > 0) & (df['avg_score'] > 0)]

fig, ax = plt.subplots(figsize=(8, 6))
for platform, color, marker in [('cpu', '#2196F3', 'o'), ('gpu', '#FF9800', 's')]:
    data = wt_scores[wt_scores.platform == platform]
    ax.scatter(data['wall_time_h'], data['avg_score'], c=color, marker=marker,
               alpha=0.7, s=60, label=f'{platform.upper()} (n={len(data)})')

ax.set_xlabel('Wall Time (hours)')
ax.set_ylabel('Peer Review Score')
ax.set_title('Kimi — Score vs Wall Time')
ax.axhline(y=8, color='green', linestyle='--', alpha=0.3, label='Accept')
ax.legend()

# Correlation
if len(wt_scores) > 2:
    corr = wt_scores[['wall_time_h', 'avg_score']].corr().iloc[0, 1]
    ax.text(0.05, 0.95, f'r = {corr:.2f}', transform=ax.transAxes, fontsize=11)

plt.tight_layout()
plt.savefig('kimi_score_vs_time.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Summary Statistics

In [ ]:
print('=== KIMI RESEARCHER SUMMARY ===')
print(f'Total trials: {len(df)}')
print(f'  CPU: {len(df[df.platform=="cpu"])} trials across {df[df.platform=="cpu"].seed.nunique()} seeds')
print(f'  GPU: {len(df[df.platform=="gpu"])} trials across {df[df.platform=="gpu"].seed.nunique()} seeds')
print()
print(f'Overall mean score: {df.avg_score.mean():.2f} ± {df.avg_score.sem():.2f}')
print(f'  CPU: {df[df.platform=="cpu"].avg_score.mean():.2f} ± {df[df.platform=="cpu"].avg_score.sem():.2f}')
print(f'  GPU: {df[df.platform=="gpu"].avg_score.mean():.2f} ± {df[df.platform=="gpu"].avg_score.sem():.2f}')
print()
print(f'Best score: {df.avg_score.max():.1f}')
best = df[df.avg_score == df.avg_score.max()].iloc[0]
print(f'  Seed: {best.seed} ({best.platform}), Trial {best.trial}')
print()
print(f'Papers scoring >= 4: {len(df[df.avg_score >= 4])} / {len(df)} ({100*len(df[df.avg_score >= 4])/len(df):.0f}%)')
print(f'Papers scoring >= 5: {len(df[df.avg_score >= 5])} / {len(df)} ({100*len(df[df.avg_score >= 5])/len(df):.0f}%)')
print(f'Papers scoring >= 6: {len(df[df.avg_score >= 6])} / {len(df)} ({100*len(df[df.avg_score >= 6])/len(df):.0f}%)')

## 12. Self-Review Score Trends

How do self-review scores change across successive reviews within the same trial?

In [ ]:
import json as _json
from collections import defaultdict

# Collect self-review score sequences from tracker data
sequences = defaultdict(lambda: defaultdict(list))

for tracker_file in sorted((BASE / "results" / "kimi").glob("*/trial*/tracker/tracker.json")):
    try:
        tracker = _json.loads(tracker_file.read_text())
        trial_key = "/".join(tracker_file.parts[-4:-2])
        for a in tracker.get("actions", []):
            stage = a.get("stage", "")
            if "self_review" not in stage: continue
            details = a.get("details", "")
            if "score=" in details:
                score = float(details.split("score=")[1].split(",")[0])
                sequences[stage][trial_key].append(score)
    except: pass

for tracker_file in sorted((BASE / "outputs" / "kimi").glob("run_*/*/tracker.json")):
    try:
        tracker = _json.loads(tracker_file.read_text())
        parts = tracker_file.parts
        trial_key = f"{parts[-2]}_gpu/run_{parts[-3].split(chr(95))[1]}"
        for a in tracker.get("actions", []):
            stage = a.get("stage", "")
            if "self_review" not in stage: continue
            details = a.get("details", "")
            if "score=" in details:
                score = float(details.split("score=")[1].split(",")[0])
                sequences[stage][trial_key].append(score)
    except: pass

# Plot trends
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
gates = ["self_review_idea", "self_review_experiment", "self_review_paper"]
titles = ["Idea Self-Review", "Experiment Self-Review", "Paper Self-Review"]
colors = ["#2196F3", "#FF9800", "#4CAF50"]

for ax, gate, title, color in zip(axes, gates, titles, colors):
    all_seqs = sequences.get(gate, {})
    if not all_seqs:
        ax.set_title(f"{title} (no data)")
        continue
    max_len = max(len(s) for s in all_seqs.values())
    
    # Plot individual trajectories (light)
    for seq in all_seqs.values():
        ax.plot(range(1, len(seq)+1), seq, color=color, alpha=0.15, linewidth=1)
    
    # Plot mean trajectory (bold)
    means = []
    sems = []
    ns = []
    for pos in range(max_len):
        scores = [s[pos] for s in all_seqs.values() if pos < len(s)]
        if scores:
            means.append(np.mean(scores))
            sems.append(np.std(scores) / np.sqrt(len(scores)))
            ns.append(len(scores))
    
    positions = range(1, len(means)+1)
    ax.errorbar(positions, means, yerr=sems, color=color, linewidth=2.5,
                marker="o", capsize=4, label="Mean ± SE")
    ax.axhline(y=8, color="green", linestyle="--", alpha=0.3, label="Pass (8)")
    ax.axhline(y=6, color="orange", linestyle="--", alpha=0.3, label="Pass (6)")
    
    # Annotate N at each position
    for x, m, n in zip(positions, means, ns):
        ax.annotate(f"n={n}", xy=(x, m), xytext=(x, m-0.8), fontsize=8, ha="center")
    
    ax.set_xlabel("Review Number")
    ax.set_title(title)
    ax.set_ylim(0, 10)
    ax.legend(fontsize=8)

axes[0].set_ylabel("Self-Review Score")
fig.suptitle("Kimi — Self-Review Score Trends (individual trials + mean)", fontsize=13)
plt.tight_layout()
plt.savefig("kimi_self_review_trends.png", dpi=150, bbox_inches="tight")
plt.show()

# Print improvement stats
for gate, title in zip(gates, titles):
    all_seqs = sequences.get(gate, {})
    multi = [(k, s) for k, s in all_seqs.items() if len(s) >= 2]
    if not multi: continue
    improving = sum(1 for _, s in multi if s[-1] > s[0])
    declining = sum(1 for _, s in multi if s[-1] < s[0])
    flat = sum(1 for _, s in multi if s[-1] == s[0])
    print(f"{title}: {len(multi)} multi-review trials — "
          f"{improving} improving ({100*improving/len(multi):.0f}%), "
          f"{declining} declining ({100*declining/len(multi):.0f}%), "
          f"{flat} flat ({100*flat/len(multi):.0f}%)")

## 13. Experiment Failure Analysis

What goes wrong? Categorize all reviewer-reported weaknesses across all trials.

In [ ]:
import refrom collections import Counterweaknesses_raw = []for reviews_file in list((BASE / 'results' / 'kimi').glob('*/trial*/code/idea_*/reviews.json')) + \                    list((BASE / 'outputs' / 'kimi').glob('run_*/*/idea_*/reviews.json')) + \                    list((BASE / 'outputs' / 'kimi').glob('*_2026*/idea_*/reviews.json')):    try:        reviews = json.loads(reviews_file.read_text())        for rev in reviews.get('reviews', []):            for w in rev.get('weaknesses', []):                weaknesses_raw.append(w)    except: passfor f in list((BASE / 'outputs' / 'kimi').glob('run_*/*/idea_*_review_logs/*_stdout.txt')) + \         list((BASE / 'outputs' / 'kimi').glob('*_2026*/idea_*_review_logs/*_stdout.txt')):    try:        content = f.read_text(errors='replace')        for line in content.split('\n'):            text = ''            try:                obj = json.loads(line.strip())                if isinstance(obj, dict):                    item = obj.get('item', {})                    if item.get('type') == 'agent_message': text = item.get('text', '')                    elif obj.get('role') == 'assistant':                        for b in obj.get('content', []):                            if isinstance(b, dict) and b.get('type') == 'text': text = b.get('text', '')            except: text = line.strip()            if not text or 'weaknesses' not in text.lower(): continue            brace = 0; start = None            for i, c in enumerate(text):                if c == '{':                    if brace == 0: start = i                    brace += 1                elif c == '}':                    brace -= 1                    if brace == 0 and start is not None:                        candidate = text[start:i+1]                        if 'weaknesses' in candidate:                            try:                                cleaned = re.sub(r',\s*}', '}', re.sub(r',\s*]', ']', candidate))                                data = json.loads(cleaned)                                for w in data.get('weaknesses', []):                                    if w not in weaknesses_raw: weaknesses_raw.append(w)                            except: pass                        start = None    except: passweaknesses_raw = list(dict.fromkeys(weaknesses_raw))print(f'Total unique weakness items: {len(weaknesses_raw)}')# Categorize — order matters! Most specific first, broadest lastcategories = Counter()cat_rules = [    # Integrity issues (most specific, check first)    ('Fabricated/hardcoded results', ['fabricat', 'hardcod', 'manually written', 'invented', 'fake result']),    ('Results mismatch', ['mismatch', 'inconsisten', 'contradict', 'discrepan', 'does not match', 'dont match', "don't match", 'differ from']),        # Execution failures      ('Code errors/crashes', ['crash', 'traceback', 'runtimeerror', 'attributeerror', 'valueerror', 'keyerror', 'killed', 'oom', 'memory error']),    ('Missing experiments', ['not run', 'never ran', 'not executed', 'not implemented', 'not completed', 'absent', 'no results']),    ('Incomplete experiments', ['incomplete', 'partial', 'in progress', 'only ran', 'only completed']),        # Rigor issues    ('Weak/missing baselines', ['baseline', 'comparison', 'no strong', 'unfair comparison']),    ('Missing ablations', ['ablation']),    ('Plan non-compliance', ['plan', 'not follow', 'deviat']),    ('No error bars/seeds', ['error bar', 'confidence interval', 'variance', 'std', 'single seed', 'one seed']),    ('Missing statistical tests', ['statistical', 'p-value', 'significance test', 'hypothesis test']),        # Quality issues    ('Novelty concerns', ['novelty', 'novel', 'prior work', 'already exist', 'incremental', 'similar to', 'closely related']),    ('Unsupported claims', ['claim', 'not support', 'overstate', 'overclaim', 'exaggerat']),    ('Reference issues', ['reference', 'citation', 'cite', 'bibliography', 'bibtex']),    ('Failed success criteria', ['target', 'success criteria', 'not achieved', 'falls short', 'below target']),    ('Weak results/performance', ['accuracy', 'performance', 'f1', 'auc', 'low score', 'poor result', 'underperform', 'below baseline']),        # Scope issues    ('Limited scope/scale', ['scope', 'narrow', 'only one dataset', 'single dataset', 'toy', 'too small']),    ('Generalization concerns', ['generali', 'transfer', 'domain shift', 'out-of-distribution', 'robustness']),    ('Insufficient coverage', ['only', 'just one', 'few', 'insufficient', 'not enough', 'too few', 'limited to']),        # Method issues    ('Methodology concerns', ['heuristic', 'ad hoc', 'unsound', 'questionable', 'flawed approach']),    ('Training issues', ['overfit', 'epoch', 'convergence', 'learning rate', 'gradient', 'fine-tun', 'training']),    ('Model/architecture issues', ['architecture', 'layer', 'parameter', 'embedding', 'frozen', 'degenerate', 'model']),    ('Simulation/not real', ['simulat', 'synthetic', 'not a real', 'proxy']),        # Documentation issues      ('Evaluation methodology', ['metric', 'evaluation', 'measure', 'benchmark']),    ('Reproducibility issues', ['reproduci', 'hyperparameter', 'config', 'implementation detail']),    ('Writing/clarity', ['writing', 'clarity', 'presentation', 'unclear', 'confusing', 'typo', 'poorly written']),    ('Dataset issues', ['dataset', 'data quality', 'data split', 'leakage', 'preprocessing']),        # Resource issues    ('Time/budget issues', ['timeout', 'budget', 'time limit', 'infeasible', 'too slow']),    ('Practical limitations', ['overhead', 'latency', 'throughput', 'practical barrier', 'scalab', 'cost']),        # Meta issues    ('Weak/missing theory', ['theoretical', 'theory', 'proof', 'bound', 'no formal', 'no analysis']),    ('Insufficient contribution', ['negative result', 'workshop', 'does not advance', 'too incremental', 'contribution']),    ('Critical flaws', ['critical', 'severe', 'fundamental', 'broken', 'catastrophic']),        # Catch-alls (broadest, last)    ('Missing/empty items', ['missing', 'empty', 'absent', 'no ', 'lack']),    ('Experiment issues', ['experiment', 'test', 'run', 'output', 'log', 'pipeline', 'code']),    ('Limited/weak', ['limited', 'weak', 'small', 'narrow', 'poor']),]for fb in weaknesses_raw:    fb_lower = fb.lower()    matched = False    for cat, keywords in cat_rules:        if any(kw in fb_lower for kw in keywords):            categories[cat] += 1            matched = True            break    if not matched:        categories['Other'] += 1# Merge small categories (< 1.5%) into Othertotal = sum(categories.values())min_count = max(1, int(total * 0.015))main_cats = {}other_count = categories.get('Other', 0)for cat, count in categories.items():    if cat == 'Other': continue    if count >= min_count:        main_cats[cat] = count    else:        other_count += countif other_count > 0:    main_cats['Other'] = other_count# Sort and plotcats = [c for c, _ in sorted(main_cats.items(), key=lambda x: x[1], reverse=True)]counts = [main_cats[c] for c in cats]pcts = [100*n/total for n in counts]fig, ax = plt.subplots(figsize=(12, 9))color_map = {    'Fabricated/hardcoded results': '#B71C1C', 'Results mismatch': '#D32F2F',    'Code errors/crashes': '#E53935', 'Missing experiments': '#F44336',    'Incomplete experiments': '#EF5350', 'Weak/missing baselines': '#FF9800',    'Missing ablations': '#FFA726', 'Plan non-compliance': '#FFB74D',    'No error bars/seeds': '#FFD54F', 'Novelty concerns': '#3F51B5',    'Unsupported claims': '#E91E63', 'Reference issues': '#9C27B0',    'Weak results/performance': '#FF5722', 'Limited scope/scale': '#FF7043',    'Insufficient coverage': '#FFAB91', 'Missing/empty items': '#FFCCBC',    'Methodology concerns': '#5C6BC0', 'Training issues': '#7986CB',    'Critical flaws': '#C62828', 'Experiment issues': '#42A5F5',    'Other': '#BDBDBD',}colors = [color_map.get(c, '#90CAF9') for c in cats]bars = ax.barh(range(len(cats)), pcts, color=colors, alpha=0.85)ax.set_yticks(range(len(cats)))ax.set_yticklabels(cats, fontsize=9)ax.set_xlabel('% of all weaknesses')ax.set_title(f'Kimi — Reviewer-Reported Weakness Categories ({total} items, CPU + GPU)')for bar, count, pct in zip(bars, counts, pcts):    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,            f'{count} ({pct:.0f}%)', va='center', fontsize=8)plt.tight_layout()plt.savefig('kimi_failure_categories.png', dpi=150, bbox_inches='tight')plt.show()print(f'\nOther: {main_cats.get("Other", 0)} items ({100*main_cats.get("Other",0)/total:.1f}%)')print(f'Categories shown: {len([c for c in cats if c != "Other"])}')

## 14. Programming Language Usage

What languages does Kimi use for experiments?

In [ ]:
lang_map = {    '.py': 'Python', '.sh': 'Shell', '.r': 'R', '.R': 'R', '.jl': 'Julia',    '.java': 'Java', '.cpp': 'C++', '.c': 'C', '.rs': 'Rust', '.go': 'Go',    '.js': 'JavaScript', '.ts': 'TypeScript', '.m': 'MATLAB',}from collections import Counter, defaultdictoverall = Counter()seed_trial_counts = defaultdict(list)  # normalized_seed -> [py_count_per_trial]def normalize_seed_name(raw):    """Normalize seed name: strip _gpu, _cpu, timestamps, underscores."""    s = raw    # Strip timestamp suffix (YYYYMMDD_HHMMSS)    parts = s.rsplit('_', 2)    if len(parts) >= 3 and parts[-2].isdigit() and parts[-1].isdigit():        s = '_'.join(parts[:-2])    s = (s.replace('_gpu', '').replace('_cpu', '')          .replace('_', ' ').title()          .replace('Ai For', 'AI for')          .replace('Of Learned Representations', 'of Learned Repr.')          .replace('And Benchmarks', '& Benchmarks')          .replace('And Cleaning', '& Cleaning')          .replace('In Machine Learning', 'in ML')          .replace('Representation Learning', 'Repr. Learning'))    return sdef get_trial_key(exp_dir):    """Return (normalized_seed, trial_id) for dedup."""    s = str(exp_dir)    after = s.split('/kimi/')[1].split('/')    if '/results/' in s:        return (normalize_seed_name(after[0]), after[1])    elif after[0].startswith('run_'):        return (normalize_seed_name(after[1]), after[0].replace('run_', 'trial'))    else:        return (normalize_seed_name(after[0]), 'trial1')seen_trials = set()for exp_dir in list((BASE / 'results' / 'kimi').glob('*/trial*/code/idea_*/exp')) + \               list((BASE / 'outputs' / 'kimi').glob('run_*/*/idea_*/exp')) + \               list((BASE / 'outputs' / 'kimi').glob('*_2026*/idea_*/exp')):        trial_key = get_trial_key(exp_dir)        # Normalize the seed part for dedup across sources    norm_seed = normalize_seed_name(trial_key[0])    dedup_key = (norm_seed, trial_key[1])        if dedup_key in seen_trials:        continue    seen_trials.add(dedup_key)        py_count = sum(1 for f in exp_dir.rglob('*') if f.is_file() and f.suffix == '.py')    all_count = sum(1 for f in exp_dir.rglob('*') if f.is_file() and f.suffix in lang_map)    seed_trial_counts[norm_seed].append(py_count)    overall.update(lang_map.get(f.suffix, f.suffix) for f in exp_dir.rglob('*')                    if f.is_file() and f.suffix in lang_map)print(f'Unique trials: {len(seen_trials)}')for seed in sorted(seed_trial_counts):    print(f'  {seed}: {len(seed_trial_counts[seed])} trials')fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))# Pie chartlangs = list(overall.keys())counts = [overall[l] for l in langs]colors_pie = ['#2196F3', '#FF9800', '#4CAF50', '#F44336'][:len(langs)]wedges, texts, autotexts = ax1.pie(counts, labels=langs, autopct='%1.1f%%',    startangle=90, colors=colors_pie, textprops={'fontsize': 10})for t in autotexts:    t.set_fontsize(10)ax1.set_title(f'Language Distribution ({sum(counts)} total files)', fontsize=12)# Average Python files per seed (sorted)seed_avg = {s: np.mean(trials) for s, trials in seed_trial_counts.items()}seed_se = {s: np.std(trials) / np.sqrt(len(trials)) if len(trials) > 1 else 0            for s, trials in seed_trial_counts.items()}seeds = sorted(seed_avg.keys(), key=lambda s: seed_avg[s])avgs = [seed_avg[s] for s in seeds]ses = [seed_se[s] for s in seeds]n_trials = [len(seed_trial_counts[s]) for s in seeds]y = range(len(seeds))ax2.barh(y, avgs, xerr=ses, color='#2196F3', alpha=0.8, height=0.6, capsize=3)ax2.set_yticks(y)ax2.set_yticklabels(seeds, fontsize=9)ax2.set_xlabel('Avg Python Files Per Trial (± SE)')ax2.set_title('Python Files Per Seed (averaged across trials)', fontsize=12)for j, (avg, se, n) in enumerate(zip(avgs, ses, n_trials)):    ax2.text(avg + se + 1, j, f'{avg:.0f} (n={n})', va='center', fontsize=8)plt.tight_layout()plt.savefig('kimi_language_usage.png', dpi=150, bbox_inches='tight')plt.show()print(f"\nOverall: {dict(overall.most_common())}")print(f"Avg Python files per trial across all seeds: {np.mean(list(seed_avg.values())):.0f}")

## 15. Experiment Count & Completion Rate

How many experiments does the agent plan vs actually complete?

In [ ]:
from collections import defaultdictdef normalize_seed_name(raw):    """Strip _gpu, _cpu, timestamps, then title-case."""    s = raw    parts = s.rsplit('_', 2)    if len(parts) >= 3 and parts[-2].isdigit() and parts[-1].isdigit():        s = '_'.join(parts[:-2])    return (s.replace('_gpu', '').replace('_cpu', '')             .replace('_', ' ').title()             .replace('Ai For', 'AI for')             .replace('Of Learned Representations', 'of Learned Repr.')             .replace('And Benchmarks', '& Benchmarks')             .replace('And Cleaning', '& Cleaning')             .replace('In Machine Learning', 'in ML')             .replace('Representation Learning', 'Repr. Learning'))def get_trial_key(path):    """Return (normalized_seed, trial_id) for dedup."""    s = str(path)    after = s.split('/kimi/')[1].split('/')    if '/results/' in s:        # results/kimi/causal_learning_cpu/trial1/code/idea_01/exp        return (normalize_seed_name(after[0]), after[1])    elif after[0].startswith('run_'):        # outputs/kimi/run_2/ai_for_biology/idea_01/exp        return (normalize_seed_name(after[1]), after[0].replace('run_', 'trial'))    else:        # outputs/kimi/ai_for_biology_20260321_115956/idea_01/exp        return (normalize_seed_name(after[0]), 'trial1')# Collect: per trial, only use the LAST idea directorytrial_data = {}  # (norm_seed, trial) -> (display_seed, planned, completed)for parent in sorted(list((BASE / 'results' / 'kimi').glob('*/trial*/code')) +                     list((BASE / 'outputs' / 'kimi').glob('run_*/*/')) +                     list((BASE / 'outputs' / 'kimi').glob('*_2026*/'))):    if not parent.is_dir(): continue    if parent.name == 'logs': continue        # Find idea dirs with exp/    idea_dirs = sorted([d for d in parent.iterdir()                       if d.is_dir() and d.name.startswith('idea_') and 'review' not in d.name                       and (d / 'exp').exists()])    if not idea_dirs: continue        last_idea = idea_dirs[-1]    exp_dir = last_idea / 'exp'        key = get_trial_key(last_idea)    if key in trial_data: continue        folders = [d for d in exp_dir.iterdir()               if d.is_dir() and d.name not in ('__pycache__', 'shared', '.ipynb_checkpoints')]    with_results = [d for d in folders                    if any(d.glob('results*.json')) or any(d.glob('*/results*.json'))]        trial_data[key] = (key[0], len(folders), len(with_results))# Aggregate per seedseed_planned = defaultdict(list)seed_completed = defaultdict(list)for (seed, trial), (display, planned, completed) in trial_data.items():    seed_planned[display].append(planned)    seed_completed[display].append(completed)# Verifyfor seed in sorted(seed_planned):    print(f'  {seed}: n={len(seed_planned[seed])} trials')# Build summaryseeds = sorted(seed_planned.keys(), key=lambda s: np.mean(seed_planned[s]))avg_planned = [np.mean(seed_planned[s]) for s in seeds]avg_completed = [np.mean(seed_completed[s]) for s in seeds]se_planned = [np.std(seed_planned[s]) / np.sqrt(len(seed_planned[s])) if len(seed_planned[s]) > 1 else 0 for s in seeds]se_completed = [np.std(seed_completed[s]) / np.sqrt(len(seed_completed[s])) if len(seed_completed[s]) > 1 else 0 for s in seeds]n_trials = [len(seed_planned[s]) for s in seeds]completion = [100 * c / p if p > 0 else 0 for p, c in zip(avg_planned, avg_completed)]# Print tableprint(f'\n{"Seed":<35} | {"n":>3} | {"Planned":>8} | {"Completed":>9} | {"Completion":>10}')print('-' * 35 + '-|' + '-' * 5 + '|' + '-' * 10 + '|' + '-' * 11 + '|' + '-' * 12)for s, n, p, c, pct in zip(seeds, n_trials, avg_planned, avg_completed, completion):    print(f'{s:<35} | {n:>3} | {p:>8.1f} | {c:>9.1f} | {pct:>9.0f}%')overall_p = np.mean(avg_planned)overall_c = np.mean(avg_completed)overall_pct = 100 * overall_c / overall_p if overall_p > 0 else 0print(f'{"OVERALL":<35} |     | {overall_p:>8.1f} | {overall_c:>9.1f} | {overall_pct:>9.0f}%')# Plotfig, ax = plt.subplots(figsize=(12, 7))y = np.arange(len(seeds))w = 0.35ax.barh(y - w/2, avg_planned, w, xerr=se_planned, label='Avg Planned',        color='#2196F3', alpha=0.7, capsize=3)ax.barh(y + w/2, avg_completed, w, xerr=se_completed, label='Avg Completed',        color='#4CAF50', alpha=0.7, capsize=3)ax.set_yticks(y)ax.set_yticklabels(seeds, fontsize=9)ax.set_xlabel('Avg Experiments Per Trial (± SE)')ax.set_title('Kimi — Planned vs Completed Experiments Per Seed')ax.legend(loc='lower right')for j, (pct, n) in enumerate(zip(completion, n_trials)):    x_pos = max(avg_planned[j] + se_planned[j], avg_completed[j] + se_completed[j]) + 0.3    color = '#4CAF50' if pct >= 50 else '#FF9800' if pct >= 25 else '#F44336'    ax.text(x_pos, j, f'{pct:.0f}% (n={n})', va='center', fontsize=8, color=color, fontweight='bold')plt.tight_layout()plt.savefig('kimi_experiment_completion.png', dpi=150, bbox_inches='tight')plt.show()

## 16. Per-Reviewer Dimension Breakdown

How does each reviewer score each dimension? Shows reviewer-specific biases.

In [ ]:
dimensions = ['novelty', 'soundness', 'significance', 'clarity', 'reproducibility',              'experimental_rigor', 'references', 'reference_integrity', 'results_integrity']reviewer_dim = {}for reviewer_name in ['Claude Code', 'Codex', 'Kimi Code']:    dim_means = []    for dim in dimensions:        col = f'dim_{dim}_{reviewer_name}'        if col in df.columns:            vals = df[col].dropna()            dim_means.append(vals.mean() if len(vals) > 0 else np.nan)        else:            dim_means.append(np.nan)    reviewer_dim[reviewer_name] = dim_meansfig, ax = plt.subplots(figsize=(14, 6))x = np.arange(len(dimensions))w = 0.25colors = {'Claude Code': '#2196F3', 'Codex': '#4CAF50', 'Kimi Code': '#FF9800'}for i, (name, means) in enumerate(reviewer_dim.items()):    ax.bar(x + i * w, means, w, label=name, color=colors[name], alpha=0.8)ax.set_xticks(x + w)ax.set_xticklabels([d.replace('_', '\n') for d in dimensions], fontsize=9)ax.set_ylabel('Mean Score (1-10)')ax.set_title('Kimi Papers — Per-Reviewer Dimension Scores')ax.legend()ax.axhline(y=6, color='gray', linestyle='--', alpha=0.3)ax.set_ylim(0, 10)plt.tight_layout()plt.savefig('kimi_reviewer_dimensions.png', dpi=150, bbox_inches='tight')plt.show()# Print the datarev_df = pd.DataFrame(reviewer_dim, index=dimensions)print(rev_df.round(1).to_string())

## 17. Score Distribution

How are peer review scores distributed across all trials?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))# Overallax = axes[0]ax.hist(df['avg_score'], bins=np.arange(0, 10.5, 0.5), color='#2196F3', alpha=0.7, edgecolor='white')ax.axvline(x=df['avg_score'].mean(), color='red', linestyle='--', label=f'Mean={df["avg_score"].mean():.2f}')ax.axvline(x=8, color='green', linestyle='--', alpha=0.5, label='Accept (8)')ax.set_xlabel('Peer Review Score')ax.set_ylabel('Count')ax.set_title('All Trials')ax.legend(fontsize=8)# By platformax = axes[1]for platform, color in [('cpu', '#2196F3'), ('gpu', '#FF9800')]:    scores = df[df.platform == platform]['avg_score']    ax.hist(scores, bins=np.arange(0, 10.5, 0.5), color=color, alpha=0.5,            edgecolor='white', label=f'{platform.upper()} (μ={scores.mean():.2f})')ax.set_xlabel('Peer Review Score')ax.set_title('CPU vs GPU')ax.legend(fontsize=8)# Per-reviewer individual scoresax = axes[2]for col, name, color in [('reviewer_Claude Code', 'Claude', '#2196F3'),                          ('reviewer_Codex', 'Codex', '#4CAF50'),                          ('reviewer_Kimi Code', 'Kimi', '#FF9800')]:    if col in df.columns:        scores = df[col].dropna()        ax.hist(scores, bins=np.arange(-0.5, 11.5, 1), color=color, alpha=0.4,                edgecolor='white', label=f'{name} (μ={scores.mean():.1f})')ax.set_xlabel('Individual Reviewer Score')ax.set_title('Per-Reviewer Scores')ax.legend(fontsize=8)plt.tight_layout()plt.savefig('kimi_score_distributions.png', dpi=150, bbox_inches='tight')plt.show()

## 18. Idea Self-Review Score vs Peer Review Score

Does a higher idea self-review score predict a higher peer review score?

In [ ]:
# Extract final idea self-review score from trackeridea_sr_scores = []peer_scores = []for _, row in df.iterrows():    if 'sr_self_review_idea_count' not in row or pd.isna(row.get('sr_self_review_idea_count')):        continue    # Get the last idea self-review score from tracker actions    # We stored counts but not individual scores in the df — let's recalculate    pass# Alternative: scan tracker files directlydata_points = []seen_trackers = set()for tracker_file in list((BASE / 'results' / 'kimi').glob('*/trial*/tracker/tracker.json')) + \                    list((BASE / 'outputs' / 'kimi').glob('run_*/*/tracker.json')):    # Dedup    path_str = str(tracker_file)    if '/results/' in path_str:        trial_id = '/'.join(path_str.split('/kimi/')[1].split('/')[:2])    else:        parts = path_str.split('/kimi/')[1].split('/')        trial_id = f"{parts[1]}/{parts[0].replace('run_', 'trial')}"    if trial_id in seen_trackers: continue    seen_trackers.add(trial_id)        try:        tracker = json.loads(tracker_file.read_text())        actions = tracker.get('actions', [])                # Find last idea self-review score        idea_sr = None        for a in actions:            if a.get('stage') == 'self_review_idea':                details = a.get('details', '')                if 'score=' in details:                    idea_sr = float(details.split('score=')[1].split(',')[0])                # Find last experiment self-review score        exp_sr = None        for a in actions:            if a.get('stage') == 'self_review_experiment':                details = a.get('details', '')                if 'score=' in details:                    exp_sr = float(details.split('score=')[1].split(',')[0])                # Find last paper self-review score        paper_sr = None        for a in actions:            if a.get('stage') == 'self_review_paper':                details = a.get('details', '')                if 'score=' in details:                    paper_sr = float(details.split('score=')[1].split(',')[0])                # Find peer review score from the workspace        ws = tracker_file.parent        if ws.name == 'tracker':            ws = ws.parent        reviews_files = sorted(ws.glob('**/reviews.json'))        if not reviews_files:            reviews_files = sorted(ws.glob('code/idea_*/reviews.json'))        peer = None        for rf in reviews_files:            try:                r = json.loads(rf.read_text())                peer = r.get('avg_score', 0)            except: pass                if peer is not None and peer > 0:            data_points.append({                'idea_sr': idea_sr, 'exp_sr': exp_sr,                'paper_sr': paper_sr, 'peer': peer            })    except: passsr_df2 = pd.DataFrame(data_points)print(f'Trials with tracker + peer review: {len(sr_df2)}')fig, axes = plt.subplots(1, 3, figsize=(16, 5))for ax, sr_col, title in zip(axes,    ['idea_sr', 'exp_sr', 'paper_sr'],    ['Idea Self-Review', 'Experiment Self-Review', 'Paper Self-Review']):        valid = sr_df2[[sr_col, 'peer']].dropna()    if len(valid) < 3:        ax.set_title(f'{title} (insufficient data)')        continue        ax.scatter(valid[sr_col], valid['peer'], alpha=0.6, s=50, color='#2196F3')        # Fit line    if valid[sr_col].std() > 0:        z = np.polyfit(valid[sr_col], valid['peer'], 1)        p = np.poly1d(z)        x_line = np.linspace(valid[sr_col].min(), valid[sr_col].max(), 100)        ax.plot(x_line, p(x_line), 'r--', alpha=0.5)        corr = valid[[sr_col, 'peer']].corr().iloc[0, 1]        ax.text(0.05, 0.95, f'r = {corr:.2f}\nn = {len(valid)}',                transform=ax.transAxes, fontsize=10, verticalalignment='top')        ax.set_xlabel(f'{title} Score')    ax.set_ylabel('Peer Review Score')    ax.set_title(title)    ax.set_xlim(-0.5, 10.5)    ax.set_ylim(-0.5, 10.5)    ax.plot([0, 10], [0, 10], 'k--', alpha=0.2, label='y=x')    ax.legend(fontsize=8)plt.tight_layout()plt.savefig('kimi_selfrev_vs_peer.png', dpi=150, bbox_inches='tight')plt.show()

## 19. Seed Difficulty Ranking

Which research topics are easiest/hardest for Kimi?

In [ ]:
seed_stats = df.groupby('seed')['avg_score'].agg(['mean', 'std', 'count']).round(2)seed_stats['se'] = (seed_stats['std'] / np.sqrt(seed_stats['count'])).round(2)seed_stats = seed_stats.sort_values('mean', ascending=True)fig, ax = plt.subplots(figsize=(12, 7))y = range(len(seed_stats))colors = ['#F44336' if m < 3 else '#FF9800' if m < 4 else '#4CAF50' for m in seed_stats['mean']]ax.barh(y, seed_stats['mean'], xerr=seed_stats['se'], color=colors, alpha=0.8, capsize=4)ax.set_yticks(y)ax.set_yticklabels(seed_stats.index, fontsize=9)ax.set_xlabel('Mean Peer Review Score')ax.set_title('Kimi — Seed Difficulty Ranking (all trials)')ax.axvline(x=8, color='green', linestyle='--', alpha=0.3, label='Accept')ax.axvline(x=4, color='red', linestyle='--', alpha=0.3, label='Reject')for i, (_, row) in enumerate(seed_stats.iterrows()):    ax.text(row['mean'] + row['se'] + 0.1, i, f'{row["mean"]:.1f}±{row["se"]:.1f} (n={int(row["count"])})',            va='center', fontsize=8)ax.legend()plt.tight_layout()plt.savefig('kimi_seed_difficulty.png', dpi=150, bbox_inches='tight')plt.show()

## 20. Score Heatmap

Complete heatmap of all seed × trial scores.

In [ ]:
pivot = df.pivot_table(values='avg_score', index='seed', columns='trial', aggfunc='first')pivot = pivot.sort_index()fig, ax = plt.subplots(figsize=(8, 10))im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=8)ax.set_xticks(range(len(pivot.columns)))ax.set_xticklabels([f'Trial {c}' for c in pivot.columns])ax.set_yticks(range(len(pivot.index)))ax.set_yticklabels(pivot.index, fontsize=9)# Annotate cellsfor i in range(len(pivot.index)):    for j in range(len(pivot.columns)):        val = pivot.iloc[i, j]        if not np.isnan(val):            color = 'white' if val < 3 else 'black'            ax.text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=10, color=color)ax.set_title('Kimi — Peer Review Scores (Seed × Trial)')ax.set_xlabel('Trial')ax.set_ylabel('Seed')fig.colorbar(im, ax=ax, label='Score', shrink=0.6)plt.tight_layout()plt.savefig('kimi_score_heatmap.png', dpi=150, bbox_inches='tight')plt.show()